# Notebook 3: Pricing Engine, Revenue Simulation and SHAP Explanations

This notebook has 3 parts:
1. **Pricing engine** - turns predicted occupancy + competitor price into a room price (clear rule)
2. **Revenue simulation** - compares AI price with a fixed price and with "copy the competitor"
3. **SHAP explanation** - writes a plain sentence about *why* the price was suggested

It uses the XGBoost model from Notebook 2.

## Step 1: Load data and model

In [ ]:
import os, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/pricing_project'
except ImportError:
    BASE = os.environ.get('PROJECT_DIR', '.')

MODELS = f'{BASE}/models'; RESULTS = f'{BASE}/results'
df = pd.read_csv(f'{BASE}/data/clean/main_clean.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
FEATURES = joblib.load(f'{MODELS}/features.pkl')
xgb = joblib.load(f'{MODELS}/xgboost.pkl')

split = int(len(df) * 0.8)
train, test = df.iloc[:split].copy(), df.iloc[split:].copy()
test['Pred_Occupancy'] = np.clip(xgb.predict(test[FEATURES]), 0, 100)
print('Test days:', len(test))

## Part 1: The pricing engine
The rule is simple so that a small hotel owner can understand it:

**Step A - Demand price:** start from the hotel's own base rate and move it up or down depending on predicted occupancy.
`demand_price = base_rate x (1 + K x (predicted_occupancy - 60) / 100)`
(60% is a "normal" occupancy. Above 60% the price goes up, below 60% it goes down.)

**Step B - Mix with the market:** `price = W x demand_price + (1 - W) x competitor_price`

**Step C - Safety limits:** the price can never go below 90% of the base rate or above 2x the base rate, and is rounded to the nearest 50 LKR.

In [ ]:
CONFIG = {
    'normal_occupancy': 60,   # occupancy % treated as normal
    'K': 1.0,                 # how strongly demand moves the price
    'W': 0.6,                 # weight of demand price (the rest is the competitor price)
    'min_factor': 0.9,        # lowest allowed price = 0.9 x base rate
    'max_factor': 2.0,        # highest allowed price = 2.0 x base rate
    'round_to': 50,
}

def recommend_price(pred_occ, base_rate, competitor_price, cfg=CONFIG):
    demand_price = base_rate * (1 + cfg['K'] * (pred_occ - cfg['normal_occupancy']) / 100)
    price = cfg['W'] * demand_price + (1 - cfg['W']) * competitor_price
    price = np.clip(price, cfg['min_factor'] * base_rate, cfg['max_factor'] * base_rate)
    return np.round(price / cfg['round_to']) * cfg['round_to']

test['AI_Price'] = recommend_price(test.Pred_Occupancy, test.Property_Base_Rate_LKR, test.Competitor_Avg_Price_LKR)
test[['Date', 'Pred_Occupancy', 'Property_Base_Rate_LKR', 'Competitor_Avg_Price_LKR', 'AI_Price']].head(8)

## Part 2: Revenue simulation
We compare 3 pricing strategies on the **test period** (the future the model has not seen):
| Strategy | Price |
|---|---|
| Fixed price | The hotel's base rate every day (what many owners do) |
| Copy competitor | The competitor average price each day |
| AI price | Our pricing engine |

**How do we know how many guests come at a new price?** We have no real price-response data, so we use a simple assumption (price elasticity `e`):
`simulated_occupancy = real_occupancy x (price / base_rate) ^ (-e)`
If `e` is high, guests are very price-sensitive. We do not know the true value for Sri Lanka, so we test **several values** and show all of them. This is stated as a limitation in the thesis.

Revenue measure = **RevPAR** (revenue per available room) = price x occupancy.

In [ ]:
def simulate(prices, e):
    occ = np.clip(test.Historical_Occupancy_Rate * (prices / test.Property_Base_Rate_LKR) ** (-e), 0, 100)
    return (prices * occ / 100).mean()          # average RevPAR per day (LKR)

strategies = {
    'Fixed price (base rate)': test.Property_Base_Rate_LKR,
    'Copy competitor': test.Competitor_Avg_Price_LKR,
    'AI price': test.AI_Price,
}

rows = []
for e in [0.5, 0.8, 1.0, 1.2, 1.5]:
    base = simulate(strategies['Fixed price (base rate)'], e)
    for name, p in strategies.items():
        rev = simulate(p, e)
        rows.append({'elasticity_e': e, 'strategy': name, 'avg_RevPAR_LKR': round(rev), 'change_vs_fixed_%': round((rev / base - 1) * 100, 1)})
sim = pd.DataFrame(rows)
sim.to_csv(f'{RESULTS}/revenue_simulation.csv', index=False)
sim.pivot(index='elasticity_e', columns='strategy', values='change_vs_fixed_%')

**How to read this table:** numbers are the % change in revenue compared with the fixed price.
- If guests are not very price-sensitive (`e` below 1), higher prices bring more revenue.
- If guests are very price-sensitive (`e` above 1), higher prices lose guests and revenue drops. This is true for any pricing method, so be honest about it in your thesis.
- Check whether the **AI price loses less than copying the competitor** when `e` is high. That shows the AI price is safer.

In [ ]:
# Does the price follow demand? (correlation between price and real occupancy)
follow = {name: np.corrcoef(p, test.Historical_Occupancy_Rate)[0, 1] if p.std() > 0 else 0.0 for name, p in strategies.items()}
print('Correlation between price and real occupancy:')
for k, v in follow.items():
    print(f'  {k:28s} {v:.2f}')

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(test.Date, test.Property_Base_Rate_LKR, label='Fixed price', color='gray')
ax.plot(test.Date, test.Competitor_Avg_Price_LKR, label='Competitor price', color='orange', alpha=0.7)
ax.plot(test.Date, test.AI_Price, label='AI price', color='green', linewidth=2)
ax.set_title('Room price per night (LKR) - test period'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RESULTS}/price_comparison.png', dpi=150); plt.show()

## Part 3: SHAP - explain *why* the price was suggested
SHAP shows how much each input pushed the **predicted occupancy** up or down. The pricing rule then turns predicted occupancy into a price. So the explanation tells the owner: *what made demand look high or low*.

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(test[FEATURES])

shap.summary_plot(shap_values, test[FEATURES], show=False)
plt.tight_layout(); plt.savefig(f'{RESULTS}/shap_summary.png', dpi=150, bbox_inches='tight'); plt.show()

## Plain-language explanation function

In [ ]:
# Easy names for each input (these words appear on the website)
FRIENDLY = {
    'SLTDA_Tourist_Arrivals': 'tourist arrivals',
    'Booking_Lead_Time': 'how early guests are booking',
    'Seasonality_Index_Norm': 'the tourist season',
    'Season_Code': 'the tourist season',
    'Day_of_Week': 'the day of the week',
    'Is_Weekend': 'the weekend',
    'Month': 'the month',
    'Demand_Surge_Flag': 'a special demand surge',
    'Arrivals_Lag7': "last week's tourist arrivals",
    'Arrivals_Roll7': 'the recent trend in tourist arrivals',
    'Occupancy_Lag7': "last week's occupancy",
}
train_median = train[FEATURES].median().to_dict()

## Step 4: Save the pricing engine file (the website will use it)

In [ ]:
engine_code = '''
import numpy as np

CONFIG = %s

FRIENDLY = %s

def recommend_price(pred_occ, base_rate, competitor_price, cfg=CONFIG):
    demand_price = base_rate * (1 + cfg["K"] * (pred_occ - cfg["normal_occupancy"]) / 100)
    price = cfg["W"] * demand_price + (1 - cfg["W"]) * competitor_price
    price = np.clip(price, cfg["min_factor"] * base_rate, cfg["max_factor"] * base_rate)
    return float(np.round(price / cfg["round_to"]) * cfg["round_to"])

def explain_day(features, row, shap_row, train_median, pred_occ, price, base, comp, top_n=3):
    # join inputs that have the same easy name (for example the two season inputs)
    groups = {}
    for f, s in zip(features, shap_row):
        name = FRIENDLY[f]
        groups.setdefault(name, [0.0, f])
        groups[name][0] += float(s)
    top = sorted(groups.items(), key=lambda kv: -abs(kv[1][0]))[:top_n]
    reasons = []
    for name, (s, f) in top:
        up = s > 0
        level = "high" if row[f] > train_median[f] else "low"
        verb = "pushes demand up" if up else "pushes demand down"
        if (level == "high") == up:
            reasons.append(f"{name} is {level}, which {verb}")
        else:
            reasons.append(f"{name} {verb}")
    move = "higher than" if price > base else "lower than" if price < base else "the same as"
    return (f"Suggested price: LKR {price:,.0f} ({move} your usual LKR {base:,.0f}). "
            f"Expected occupancy: {pred_occ:.0f}%%. Why: " + "; ".join(reasons) +
            f". Competitors charge about LKR {comp:,.0f}.")
''' % (repr(CONFIG), repr(FRIENDLY))

open(f'{MODELS}/pricing_engine.py', 'w').write(engine_code)
json.dump(train_median, open(f'{MODELS}/train_median.json', 'w'))
json.dump(CONFIG, open(f'{MODELS}/pricing_config.json', 'w'))

import importlib.util
spec = importlib.util.spec_from_file_location('pricing_engine', f'{MODELS}/pricing_engine.py')
pe = importlib.util.module_from_spec(spec); spec.loader.exec_module(pe)
print('Saved engine works. Test price for 75 percent occupancy:', pe.recommend_price(75, 8000, 11000))

## Step 5: Try the explanations on 3 days (busiest, quietest, normal)

In [ ]:
idx = [test.Pred_Occupancy.values.argmax(), test.Pred_Occupancy.values.argmin(), len(test) // 2]
for i in idx:
    r = test.iloc[i]
    text = pe.explain_day(FEATURES, r, shap_values[i], train_median, r.Pred_Occupancy, r.AI_Price,
                          r.Property_Base_Rate_LKR, r.Competitor_Avg_Price_LKR)
    print(r.Date.date(), '->', text)
    print()

## Done. Saved in `models/`: `pricing_engine.py`, `pricing_config.json`, `train_median.json`. Results and charts are in `results/`.

**What to write in the thesis:** the pricing rule, the 3 strategies, the elasticity assumption and the full table, the SHAP chart, and 2-3 example explanations.

**Next: Notebook 4 / Backend - FastAPI server that uses these files (then the Next.js website).**